# 06 Error Analysis

Global error analysis for the Common Voice Yue Whisper API baseline. This notebook merges `02` predictions with `03` tone taxonomy, then exports compact breakdown tables for later dialect and multimodal notebooks.


## Setup


In [1]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

def find_project_root():
    cwd = Path.cwd().resolve()
    env_root = os.getenv("PROJECT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root))
    candidates += [cwd, *cwd.parents]
    candidates += [
        Path(r"D:/GitHub/Cantonese-Speech-AI"),
        Path(r"D:\GitHub\Cantonese-Speech-AI"),
        Path("/mnt/d/GitHub/Cantonese-Speech-AI"),
        Path("/content/Cantonese-Speech-AI"),
        Path("/content/drive/MyDrive/Cantonese-Speech-AI"),
    ]
    for candidate in candidates:
        if (candidate / "src" / "cantonese_speech_ai").exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Cannot find project root from cwd={cwd}")

ROOT = find_project_root()
os.environ["PROJECT_ROOT"] = str(ROOT)
load_dotenv(ROOT / ".env")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd

ROOT


WindowsPath('D:/GitHub/Cantonese-Speech-AI')

## Load predictions and tone taxonomy


In [2]:
prediction_path = ROOT / "outputs" / "predictions" / "whisper_api_dev_20.csv"
taxonomy_path = ROOT / "outputs" / "analysis" / "tone_error_taxonomy_dev_20.csv"

if not prediction_path.exists():
    prediction_files = sorted(
        (ROOT / "outputs" / "predictions").glob("whisper_api_dev_*.csv"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not prediction_files:
        raise FileNotFoundError("Run 02_asr_baseline.ipynb before this notebook.")
    prediction_path = prediction_files[0]

predictions = pd.read_csv(prediction_path, encoding="utf-8-sig")
taxonomy = pd.read_csv(taxonomy_path, encoding="utf-8-sig") if taxonomy_path.exists() else pd.DataFrame()

analysis = predictions.copy()
if not taxonomy.empty:
    taxonomy_cols = [
        "path",
        "error_category",
        "written_replacement_flags",
        "reference_jyutping",
        "prediction_jyutping",
        "reference_tones",
        "prediction_tones",
        "tone_count_delta",
    ]
    analysis = analysis.merge(taxonomy[taxonomy_cols], on="path", how="left")

analysis["sentence_length"] = analysis["sentence"].fillna("").str.len()
analysis["accents"] = analysis["accents"].fillna("unknown").replace("", "unknown")
analysis["age"] = analysis["age"].fillna("unknown").replace("", "unknown")
analysis["gender"] = analysis["gender"].fillna("unknown").replace("", "unknown")
analysis["error_category"] = analysis["error_category"].fillna("unclassified")
analysis["written_replacement_flags"] = analysis["written_replacement_flags"].fillna("[]")
analysis["has_written_replacement"] = analysis["written_replacement_flags"].ne("[]")
analysis["tone_count_delta"] = analysis["tone_count_delta"].fillna(0)
analysis["high_cer"] = analysis["cer"] >= 0.25

{
    "prediction_path": str(prediction_path),
    "taxonomy_path": str(taxonomy_path),
    "rows": len(analysis),
    "mean_cer": analysis["cer"].mean(),
    "mean_wer": analysis["wer"].mean(),
}


{'prediction_path': 'D:\\GitHub\\Cantonese-Speech-AI\\outputs\\predictions\\whisper_api_dev_20.csv',
 'taxonomy_path': 'D:\\GitHub\\Cantonese-Speech-AI\\outputs\\analysis\\tone_error_taxonomy_dev_20.csv',
 'rows': 20,
 'mean_cer': np.float64(0.3067809886192239),
 'mean_wer': np.float64(0.8833333333333334)}

## Overall and worst examples


In [3]:
overall = {
    "rows": len(analysis),
    "mean_cer": analysis["cer"].mean(),
    "median_cer": analysis["cer"].median(),
    "mean_wer": analysis["wer"].mean(),
    "high_cer_rate": analysis["high_cer"].mean(),
    "written_style_rate": analysis["has_written_replacement"].mean(),
}
overall


{'rows': 20,
 'mean_cer': np.float64(0.3067809886192239),
 'median_cer': np.float64(0.26785714285714285),
 'mean_wer': np.float64(0.8833333333333334),
 'high_cer_rate': np.float64(0.6),
 'written_style_rate': np.float64(0.35)}

In [4]:
analysis[
    ["path", "sentence", "prediction", "cer", "wer", "error_category", "written_replacement_flags"]
].sort_values("cer", ascending=False).head(10)


,path,sentence,prediction,cer,wer,error_category,written_replacement_flags
2,common_voice_yue_39013032.mp3,似我哋嗰度嘅,像我們那裡的,0.833333,1.0,high_cer_written_style,"['嘅->的', '哋->們', '嗰->那']"
9,common_voice_yue_39013286.mp3,仲係好窮嘅鄉下仔,還是很窮的鄉下人,0.625000,1.0,high_cer_written_style,['嘅->的']
4,common_voice_yue_39013110.mp3,輸咗唔好喊，唔好走去搵阿媽求救,"輸了不要哭,不要去找媽媽求救。",0.600000,1.0,high_cer_written_style,"['唔->不', '咗->了']"
11,common_voice_yue_39013348.mp3,急都唔急嗰一陣,急都不急過一會兒,0.571429,1.0,high_cer_written_style,['唔->不']
17,common_voice_yue_39013850.mp3,我就係問緊先生住開邊個巿？,"我正在問先生,住開邊個市?",0.416667,2.0,high_cer,[]
15,common_voice_yue_39013938.mp3,做嘢做一半,工作做一半,0.400000,1.0,high_cer,[]
10,common_voice_yue_39013715.mp3,群主博覽羣書，博採眾長,"群主博覽群書,博彩頌祥。",0.363636,1.0,high_cer,[]
3,common_voice_yue_39013090.mp3,你從來都唔參加比賽，係乜嘢驅使你嚟呢度參賽？,"你從來都不參加比賽,是什麼驅使你來這裡參賽?",0.333333,1.0,high_cer_written_style,"['唔->不', '嚟->來', '乜嘢->什麼', '呢->這']"
6,common_voice_yue_39013129.mp3,都係手寫算數,還是手寫算數,0.333333,1.0,high_cer,[]
19,common_voice_yue_39013335.mp3,見到喇，唔該晒,"見到啦,唔該曬!",0.285714,1.0,high_cer,[]


## Metadata and taxonomy breakdowns


In [5]:
def breakdown(frame, column):
    grouped = (
        frame.groupby(column, dropna=False)
        .agg(
            count=("path", "count"),
            mean_cer=("cer", "mean"),
            mean_wer=("wer", "mean"),
            high_cer_rate=("high_cer", "mean"),
            written_style_rate=("has_written_replacement", "mean"),
            mean_duration_sec=("duration_sec", "mean"),
            mean_sentence_length=("sentence_length", "mean"),
        )
        .reset_index()
        .rename(columns={column: "value"})
    )
    grouped.insert(0, "dimension", column)
    return grouped.sort_values(["count", "mean_cer"], ascending=[False, False])

breakdown_tables = [
    breakdown(analysis, "accents"),
    breakdown(analysis, "age"),
    breakdown(analysis, "gender"),
    breakdown(analysis, "error_category"),
]
error_breakdown = pd.concat(breakdown_tables, ignore_index=True)

breakdown_path = ROOT / "outputs" / "analysis" / "error_breakdown_dev_20.csv"
breakdown_path.parent.mkdir(parents=True, exist_ok=True)
error_breakdown.to_csv(breakdown_path, index=False, encoding="utf-8-sig")

{
    "breakdown_path": str(breakdown_path),
    "rows": len(error_breakdown),
}


{'breakdown_path': 'D:\\GitHub\\Cantonese-Speech-AI\\outputs\\analysis\\error_breakdown_dev_20.csv',
 'rows': 7}

In [6]:
error_breakdown


,dimension,value,count,mean_cer,mean_wer,high_cer_rate,written_style_rate,mean_duration_sec,mean_sentence_length
0,accents,unknown,20,0.306781,0.883333,0.6,0.35,3.789000,10.100000
1,age,thirties,20,0.306781,0.883333,0.6,0.35,3.789000,10.100000
2,gender,female_feminine,20,0.306781,0.883333,0.6,0.35,3.789000,10.100000
3,error_category,low_error,7,0.099529,0.595238,0.0,0.00,3.630857,9.714286
4,error_category,high_cer_written_style,6,0.535516,1.000000,1.0,1.00,4.152000,11.166667
5,error_category,high_cer,6,0.341558,1.166667,1.0,0.00,3.306000,8.333333
6,error_category,written_style_replacement,1,0.176471,0.500000,0.0,1.00,5.616000,17.000000


## Reading the results

Use `error_category` to separate language-style failures from general ASR errors. Use `accents`, `age`, and `gender` only as exploratory metadata because the current API sample is small.
